# Toxicity subtype: cheating vs interpersonal

A second pass on the reviews Gemini already labeled `harm_mention = yes`. Each one gets a `harm_type`:
`cheating` (only cheaters, hackers, unfair play), `interpersonal` (insults, harassment, toxic community, ...),
`both`, or `uncertain`.

**Nothing from `scale_up.ipynb` is touched.** This notebook only *reads* `data/gemini_labels_*.csv` and writes two NEW files:
`data/gemini_harm_type_*.jsonl` (cache) and `data/gemini_harm_type_*.csv` (result). Run the cells top to bottom.

## 1. Setup

In [ ]:
import os, sys, json, time, importlib
import pandas as pd
from dotenv import load_dotenv
from google import genai

sys.path.insert(0, "../scripts")
import harm_subtype
importlib.reload(harm_subtype)   # picks up edits to scripts/harm_subtype.py without restarting the kernel

load_dotenv("../.env")
if os.environ.get("USE_VERTEX") == "1":
    client = genai.Client(
        vertexai=True,
        project=os.environ["GOOGLE_CLOUD_PROJECT"],
        location=os.environ.get("GOOGLE_CLOUD_LOCATION", "global"),
    )
else:
    client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])

## 2. Pick the toxic reviews (read-only)

In [ ]:
GEMINI_MODEL = "gemini-3.5-flash-lite"
TEMPERATURE = 0
MODEL_TAG = GEMINI_MODEL.replace("/", "_")

SOURCE = f"../data/gemini_labels_{MODEL_TAG}.csv"       # read only, never written
CACHE  = f"../data/gemini_harm_type_{MODEL_TAG}.jsonl"  # NEW: one line per sub-labeled review
OUT    = f"../data/gemini_harm_type_{MODEL_TAG}.csv"    # NEW: result table

src = pd.read_csv(SOURCE)
ok = src[src["status"] == "ok"]
df_yes = ok[ok["harm_mention"] == "yes"].copy()
print(f"{len(ok):,} labeled reviews, {len(df_yes):,} with harm_mention = yes")
df_yes["game"].value_counts()

## 3. Run

`LIMIT = 20` is a quick dry run. When the 20 look right (see section 5), set `LIMIT = None` and run this cell again:
it skips what is already saved and only does the rest. Every review is saved the moment it finishes, so you can stop and resume.

In [ ]:
LIMIT = 20   # None = all toxic reviews

with open("../prompts/harm_subtype.txt", "r") as f:
    SYSTEM_SUB = f.read()

run = harm_subtype.SubtypeRun(
    df_yes, client, GEMINI_MODEL, SYSTEM_SUB, CACHE,
    temperature=TEMPERATURE, workers=8, limit=LIMIT,
)
run.start()

## 4. Save the result table (new file)

In [ ]:
out = run.export(OUT)
out["harm_type"].value_counts(dropna=False)

## 5. Read a few examples

Check these before trusting the numbers.

In [ ]:
pd.set_option("display.max_colwidth", 160)
done = out.dropna(subset=["harm_type"])
for t in ["cheating", "interpersonal", "both", "uncertain"]:
    s = done[done["harm_type"] == t]
    if len(s):
        print(f"\n=== {t} ({len(s):,}) ===")
        for _, r in s.sample(min(5, len(s)), random_state=1).iterrows():
            print("-", r["game"], "|", str(r["review"])[:170].replace("\n", " "))
            print("    ->", r["harm_type_reason"])

## 6. Summary

Share of ALL labeled reviews (not only the toxic ones), per game. "Any interpersonal" = `interpersonal` + `both`.
Only meaningful after the full run (`LIMIT = None`).

In [ ]:
done = out.dropna(subset=["harm_type"])
tot = ok["game"].value_counts()
tab = pd.crosstab(done["game"], done["harm_type"])
for c in ["cheating", "interpersonal", "both", "uncertain"]:
    if c not in tab: tab[c] = 0
res = pd.DataFrame({
    "labeled reviews": tot,
    "toxic (yes)": tab.sum(axis=1),
    "cheating only %": (tab["cheating"] / tot * 100).round(1),
    "any interpersonal %": ((tab["interpersonal"] + tab["both"]) / tot * 100).round(1),
    "both %": (tab["both"] / tot * 100).round(1),
    "uncertain %": (tab["uncertain"] / tot * 100).round(1),
})
res.loc["All three"] = [
    tot.sum(), tab.sum().sum(),
    round(tab["cheating"].sum() / tot.sum() * 100, 1),
    round((tab["interpersonal"].sum() + tab["both"].sum()) / tot.sum() * 100, 1),
    round(tab["both"].sum() / tot.sum() * 100, 1),
    round(tab["uncertain"].sum() / tot.sum() * 100, 1),
]
res